In [1]:
import os, sys

root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

print("Project root:", root_dir)

Project root: c:\Users\NoteBook\Desktop\programing\Computer Vision\satellite-to-map-image-translation-gan


In [ ]:
import torch
import os
import matplotlib.pyplot as plt
import yaml
import subprocess
from utils.helpers import visualize_image

%matplotlib inline

with open('config.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = os.path.join(config['outputs']['checkpoints_dir'] , 'generator_best.pth')
train_history_path = os.path.join(config['outputs']['logging_dir'] , 'history.npy') 


In [ ]:
import numpy as np

history = np.load(train_history_path, allow_pickle=True).item()

train_G_loss = history['train_G_loss']
train_D_loss = history['train_D_loss']
val_loss = history['val_loss']
val_PSNR = history['val_PSNR']
val_SSIM = history['val_SSIM']
epochs = range(1, len(train_G_loss)+1)


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(epochs, train_G_loss, label='Train Generator Loss')
plt.plot(epochs, val_loss, label='Validation L1 Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plt.plot(epochs, val_PSNR, label='Validation PSNR', color='b')
plt.xlabel('Epoch')
plt.ylabel('PSNR')
plt.title('Validation PSNR over Epochs')
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(epochs, val_SSIM, label='Validation SSIM', color='g')
plt.xlabel('Epoch')
plt.ylabel('SSIM')
plt.title('Validation SSIM over Epochs')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
result = subprocess.run(
    ["python", "evaluation/metrics.py"], capture_output=True, text=True
)
print(result.stdout)


In [ ]:
from models.generator import GeneratorUNet
from utils.dataloader import get_dataloader

model = GeneratorUNet(in_channels=3, out_channels=3, base_filters=32).to(device)
model.load_state_dict(torch.load(checkpoint_path))
model.eval()

test_loader = get_dataloader(config['data_dir'], split='test', batch_size=5, shuffle=False)

satellite, real_map = next(iter(test_loader))
satellite, real_map = satellite.to(device), real_map.to(device)

with torch.no_grad():
    fake_map = model(satellite)

sat_grid = visualize_image(satellite, num_images=5)
fake_grid = visualize_image(fake_map, num_images=5)
real_grid = visualize_image(real_map, num_images=5)

fig, ax = plt.subplots(1, 3, figsize=(15,5))
ax[0].imshow(sat_grid.permute(1,2,0))
ax[0].set_title('Satellite')
ax[1].imshow(fake_grid.permute(1,2,0))
ax[1].set_title('Generated Map')
ax[2].imshow(real_grid.permute(1,2,0))
ax[2].set_title('Ground Truth')
for a in ax:
    a.axis('off')
plt.show()


In [ ]:
from utils.helpers import calculate_ssim, calculate_psnr

ssim_values = [calculate_ssim(f, r) for f, r in zip(fake_map, real_map)]
psnr_values = [calculate_psnr(f, r) for f, r in zip(fake_map, real_map)]

min_idx = np.argmin(ssim_values)
fig, ax = plt.subplots(1,3, figsize=(12,4))
ax[0].imshow(satellite[min_idx].cpu().permute(1,2,0))
ax[0].set_title('Satellite')
ax[1].imshow(fake_map[min_idx].cpu().permute(1,2,0))
ax[1].set_title(f'Generated Map\nSSIM={ssim_values[min_idx]:.3f}')
ax[2].imshow(real_map[min_idx].cpu().permute(1,2,0))
ax[2].set_title('Ground Truth')
for a in ax:
    a.axis('off')
plt.show()
